# SignSpeak Universal Prototype — Part 2: Model Training & ONNX Export
**Version:** 1.0 (Phase 0 Model Training)
**Target Hardware:** NVIDIA GeForce RTX 4050 GPU (6 GB VRAM)
**Architecture:** Hybrid ST-GCN + Multi-Head Self-Attention + BiGRU + Temporal Fusion

## Step 1: Environment & GPU Verification

In [12]:
import os
import sys
import json
import time
import random
import numpy as np
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import onnxruntime as ort

BASE_DIR = Path(r"d:\finalspeak")
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Set seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Target GPU: {torch.cuda.get_device_name(0)}")

PyTorch Version: 2.5.1+cu121
CUDA Available: True
Target GPU: NVIDIA GeForce RTX 4050 Laptop GPU


## Step 2: Load Manifest & Build Class Mapping

In [13]:
manifest_path = PROCESSED_DIR / "manifest.json"
with open(manifest_path, 'r', encoding='utf-8') as f:
    manifest_items = json.load(f)

unique_glosses = sorted(list(set(item['gloss'] for item in manifest_items)))
num_classes = len(unique_glosses)
gloss2idx = {gloss: idx for idx, gloss in enumerate(unique_glosses)}
idx2gloss = {idx: gloss for idx, gloss in enumerate(unique_glosses)}

print(f"Total Processed Clips: {len(manifest_items)}")
print(f"Total Gloss Classes: {num_classes}")

class_mapping_path = MODELS_DIR / "class_mapping.json"
with open(class_mapping_path, 'w', encoding='utf-8') as f:
    json.dump({"gloss2idx": gloss2idx, "idx2gloss": idx2gloss}, f, indent=2)
print(f"Saved Class Mapping to {class_mapping_path}")

Total Processed Clips: 6322
Total Gloss Classes: 364
Saved Class Mapping to d:\finalspeak\models\class_mapping.json


## Step 3: Dataset Splitting & PyTorch DataLoader Setup

In [14]:
random.shuffle(manifest_items)
n_total = len(manifest_items)
n_train = int(0.8 * n_total)
n_val = int(0.1 * n_total)

train_items = manifest_items[:n_train]
val_items = manifest_items[n_train:n_train + n_val]
test_items = manifest_items[n_train + n_val:]

print(f"Train Clips: {len(train_items)} | Val Clips: {len(val_items)} | Test Clips: {len(test_items)}")

class LandmarkAugmenter:
    @staticmethod
    def apply_augmentation(tensor):
        augmented = tensor.copy()
        if np.random.rand() < 0.3:
            noise = np.random.normal(0, 0.01, size=augmented.shape).astype(np.float32)
            augmented += noise
        if np.random.rand() < 0.2:
            augmented[:, :63] = 0.0
        if np.random.rand() < 0.2:
            augmented[:, 63:126] = 0.0
        return augmented

class SignLanguageDataset(Dataset):
    def __init__(self, manifest_items, processed_dir, gloss2idx, is_train=False):
        self.items = manifest_items
        self.processed_dir = Path(processed_dir)
        self.gloss2idx = gloss2idx
        self.is_train = is_train

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        file_path = self.processed_dir / item['file']
        with np.load(file_path) as data:
            features = data['features'].astype(np.float32)
        if self.is_train:
            features = LandmarkAugmenter.apply_augmentation(features)
        label = self.gloss2idx[item['gloss']]
        return torch.from_numpy(features), torch.tensor(label, dtype=torch.long)

batch_size = 32
train_dataset = SignLanguageDataset(train_items, PROCESSED_DIR, gloss2idx, is_train=True)
val_dataset = SignLanguageDataset(val_items, PROCESSED_DIR, gloss2idx, is_train=False)
test_dataset = SignLanguageDataset(test_items, PROCESSED_DIR, gloss2idx, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Train Clips: 5057 | Val Clips: 632 | Test Clips: 633


## Step 4: Hybrid ST-GCN + Attention Model Definition

In [15]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim=256, num_heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        return self.norm(x + attn_out)

class SignHybridSTGCN(nn.Module):
    def __init__(self, in_features=856, hidden_dim=256, num_classes=364, num_heads=4):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.spatial_conv = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )
        self.self_attention = MultiHeadSelfAttention(embed_dim=hidden_dim, num_heads=num_heads)
        self.bigru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim // 2,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.2
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        B, T, F_dim = x.shape
        x_flat = x.view(B * T, F_dim)
        x_proj = self.input_proj(x_flat).view(B, T, -1)
        x_conv = self.spatial_conv(x_proj.transpose(1, 2)).transpose(1, 2)
        x_attn = self.self_attention(x_conv)
        gru_out, _ = self.bigru(x_attn)
        mean_pool = torch.mean(gru_out, dim=1)
        max_pool, _ = torch.max(gru_out, dim=1)
        pooled = (mean_pool + max_pool) / 2.0
        return self.classifier(pooled)

model = SignHybridSTGCN(in_features=856, hidden_dim=256, num_classes=num_classes).to(device)
print(f"Model Parameter Count: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Model Parameter Count: 1,353,708


## Step 5: Model Training Loop (RTX 4050 FP16 AMP Acceleration)

In [16]:
def calculate_accuracy(output, target, topk=(1, 5)):
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)
        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))
        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size).item())
        return res

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
epochs = 40
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

best_val_acc = 0.0
best_model_path = MODELS_DIR / "prototype_stgcn.pth"

for epoch in range(1, epochs + 1):
    model.train()
    train_loss, train_top1, train_top5 = 0.0, 0.0, 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(inputs)
            loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        top1, top5 = calculate_accuracy(outputs, targets, topk=(1, 5))
        train_loss += loss.item() * inputs.size(0)
        train_top1 += top1 * inputs.size(0)
        train_top5 += top5 * inputs.size(0)
    scheduler.step()
    train_loss /= len(train_dataset)
    train_top1 /= len(train_dataset)
    train_top5 /= len(train_dataset)

    model.eval()
    val_loss, val_top1, val_top5 = 0.0, 0.0, 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(inputs)
                loss = criterion(outputs, targets)
            top1, top5 = calculate_accuracy(outputs, targets, topk=(1, 5))
            val_loss += loss.item() * inputs.size(0)
            val_top1 += top1 * inputs.size(0)
            val_top5 += top5 * inputs.size(0)
    val_loss /= len(val_dataset)
    val_top1 /= len(val_dataset)
    val_top5 /= len(val_dataset)

    if val_top5 > best_val_acc:
        best_val_acc = val_top5
        torch.save(model.state_dict(), best_model_path)

    if epoch % 5 == 0 or epoch == epochs:
        print(f"Epoch {epoch:02d}/{epochs:02d} | Train Loss: {train_loss:.4f} | Train Top-5: {train_top5:.2f}% || Val Loss: {val_loss:.4f} | Val Top-5: {val_top5:.2f}%")

print(f"Training Complete! Best Val Top-5 Accuracy: {best_val_acc:.2f}%")

C:\Users\Khaja Naseeruddin\AppData\Local\Temp\ipykernel_13480\3235851682.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
C:\Users\Khaja Naseeruddin\AppData\Local\Temp\ipykernel_13480\3235851682.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
C:\Users\Khaja Naseeruddin\AppData\Local\Temp\ipykernel_13480\3235851682.py:49: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Epoch 05/40 | Train Loss: 5.1608 | Train Top-5: 5.99% || Val Loss: 5.1233 | Val Top-5: 4.59%
Epoch 10/40 | Train Loss: 4.9574 | Train Top-5: 11.55% || Val Loss: 4.9174 | Val Top-5: 12.34%
Epoch 15/40 | Train Loss: 4.6402 | Train Top-5: 19.38% || Val Loss: 4.5737 | Val Top-5: 20.25%
Epoch 20/40 | Train Loss: 4.4038 | Train Top-5: 25.03% || Val Loss: 4.3370 | Val Top-5: 28.64%
Epoch 25/40 | Train Loss: 4.2594 | Train Top-5: 30.93% || Val Loss: 4.2560 | Val Top-5: 28.16%
Epoch 30/40 | Train Loss: 4.1442 | Train Top-5: 34.07% || Val Loss: 4.1314 | Val Top-5: 32.91%
Epoch 35/40 | Train Loss: 4.0661 | Train Top-5: 37.49% || Val Loss: 4.0990 | Val Top-5: 35.28%
Epoch 40/40 | Train Loss: 4.0306 | Train Top-5: 38.92% || Val Loss: 4.0827 | Val Top-5: 34.49%
Training Complete! Best Val Top-5 Accuracy: 36.08%


## Step 6: Test Evaluation & ONNX Model Export

In [17]:
# Load best model
model.load_state_dict(torch.load(best_model_path))
model.eval()
test_top1, test_top5 = 0.0, 0.0
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        top1, top5 = calculate_accuracy(outputs, targets, topk=(1, 5))
        test_top1 += top1 * inputs.size(0)
        test_top5 += top5 * inputs.size(0)

print(f"Final Test Top-1 Accuracy: {test_top1 / len(test_dataset):.2f}%")
print(f"Final Test Top-5 Accuracy: {test_top5 / len(test_dataset):.2f}%")

# Export to ONNX
onnx_path = MODELS_DIR / "prototype_stgcn.onnx"
dummy_input = torch.randn(1, 30, 856, device=device)
torch.onnx.export(
    model, dummy_input, onnx_path, export_params=True, opset_version=14,
    do_constant_folding=True, input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
print(f"Exported ONNX Binary to {onnx_path}")

C:\Users\Khaja Naseeruddin\AppData\Local\Temp\ipykernel_13480\741579711.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path)

Final Test Top-1 Accuracy: 12.16%
Final Test Top-5 Accuracy: 34.44%
Exported ONNX Binary to d:\finalspeak\models\prototype_stgcn.onnx
